# NB-A2: Bidirectional GRU + Temporal Attention - Predictive Cardiac Intelligence


## Objectives
1. **Temporal Sequence Modeling**: Operate on 8-beat windows (temporal context)
2. **Bidirectional GRU**: Capture past and future context within the sequence
3. **Multi-Head Attention**: Focus on the most clinically critical beats
4. **Strict Inter-Patient Validation**: Train {100-108}, Test {200, 201}


###  Setting the Stage: Sequences & Predicting Trends
In this notebook, I'm shifting focus from individual heartbeats to sequences. I'm importing PyTorch and checking my CUDA device again. The goal here is 'Temporal Intelligence'—understanding how heart rhythms evolve over a short window of time.

In [1]:
# Cell 1: Dependency Verification
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score
import wfdb, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__} | Device: {device}")


PyTorch: 2.5.1+cu121 | Device: cuda


###  Preparing the Sequences: Data Loader
Here, I'm setting up a sequence data loader. Instead of looking at one single heartbeat, I'm taking a sequence of 8 beats. This helps the model see the context before a potential arrhythmia occurs. I'm still using the strict Inter-Patient split to keep the evaluation honest.

In [2]:
# Cell 2: Temporal Sequence Data Loading
WINDOW_SIZE   = 300
SEQ_LEN       = 8
BATCH_SIZE    = 32
TRAIN_RECORDS = ['100', '101', '102', '103', '106', '108']
TEST_RECORDS  = ['200', '201']
LABEL_MAP     = {'N': 0, 'L': 1, 'R': 2, 'V': 3}
CLASS_NAMES   = ['Normal (N)', 'LBBB (L)', 'RBBB (R)', 'PVC (V)']

def load_beat_sequences(record_ids, window_size=300, seq_len=8):
    X_all, y_all = [], []
    for rid in record_ids:
        try:
            record = wfdb.rdrecord(rid, pn_dir='mitdb')
            ann    = wfdb.rdann(rid, extension='atr', pn_dir='mitdb')
            signal = record.p_signal[:, 0]
            beats, labels = [], []
            for i, peak in enumerate(ann.sample):
                if (ann.symbol[i] in LABEL_MAP and peak > window_size//2 and peak < len(signal)-window_size//2):
                    beats.append(signal[peak-window_size//2 : peak+window_size//2])
                    labels.append(LABEL_MAP[ann.symbol[i]])
            
            if len(beats) >= seq_len:
                for i in range(len(beats) - seq_len):
                    X_all.append(beats[i : i + seq_len])
                    y_all.append(labels[i + seq_len - 1])
                print(f"  Record {rid}: {len(beats)} beats -> {len(beats)-seq_len} sequences")
            else:
                print(f"  [SKIP] Record {rid}: Too few beats ({len(beats)})")
        except Exception as e:
            print(f"  [WARN] {rid} failed: {e}")
    
    if not X_all:
        return np.zeros((0, seq_len, window_size), dtype=np.float32), np.zeros((0,), dtype=np.int64)
    return np.array(X_all, dtype=np.float32), np.array(y_all)

print("Loading TRAINING sequences...")
X_train, y_train = load_beat_sequences(TRAIN_RECORDS)
print(f"\nLoading TEST sequences...")
X_test, y_test   = load_beat_sequences(TEST_RECORDS)

print(f"\nTraining: {X_train.shape} | Test: {X_test.shape}")

if X_train.shape[0] > 0:
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)), 
                             batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)), 
                             batch_size=BATCH_SIZE)
    print("DataLoaders created successfully.")
else:
    print("CRITICAL ERROR: No training data loaded. Check MIT-BIH records.")


Loading TRAINING sequences...
  Record 100: 2238 beats -> 2230 sequences
  Record 101: 1859 beats -> 1851 sequences
  Record 102: 103 beats -> 95 sequences
  Record 103: 2081 beats -> 2073 sequences
  Record 106: 2027 beats -> 2019 sequences
  Record 108: 1755 beats -> 1747 sequences

Loading TEST sequences...
  Record 200: 2568 beats -> 2560 sequences
  Record 201: 1823 beats -> 1815 sequences

Training: (10015, 8, 300) | Test: (4375, 8, 300)
DataLoaders created successfully.


###  The Predictive Brain: BiGRU + Temporal Attention
Now for the model architecture. I'm using a Bidirectional Gated Recurrent Unit (BiGRU). It reads the sequence of beats both forwards and backwards to capture temporal relationships. I've also implemented a 'Temporal Attention' mechanism, which lets the model decide which beats in that 8-beat sequence are most critical for the final classification.

In [3]:
# Cell 3: BiGRU Architecture
class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        return self.norm(x + attn_out)

class BiGRU_TemporalModel(nn.Module):
    def __init__(self, window_size=300, hidden_size=128, num_classes=4):
        super().__init__()
        self.beat_encoder = nn.Sequential(
            nn.Conv1d(1, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, 5, padding=2), nn.ReLU(), nn.MaxPool1d(4), nn.Flatten(),
            nn.Linear(32 * (window_size // 16), hidden_size)
        )
        self.bigru = nn.GRU(hidden_size, hidden_size, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.attention = TemporalAttention(embed_dim=hidden_size * 2)
        self.classifier = nn.Sequential(nn.Linear(hidden_size * 2, 64), nn.ReLU(), nn.Dropout(0.4), nn.Linear(64, num_classes))
    def forward(self, x):
        b, s, w = x.shape
        enc = self.beat_encoder(x.view(b*s, 1, w)).view(b, s, -1)
        gru_out, _ = self.bigru(enc)
        att = self.attention(gru_out)
        return self.classifier(att.mean(dim=1))

model_a2 = BiGRU_TemporalModel(window_size=WINDOW_SIZE).to(device)
print(f"BiGRU Params: {sum(p.numel() for p in model_a2.parameters()):,}")


BiGRU Params: 851,556


###  Training the Temporal Model
Time to train the BiGRU! I've added class weights here because the dataset is imbalanced. This ensures that the model doesn't ignore the rarer, more dangerous rhythms. I'm tracking the loss and accuracy over 10 epochs. It's exciting to see the accuracy climb as the model learns the temporal patterns!

In [4]:
# Cell 4: Training Loop
class_weights = torch.tensor([0.5, 1.5, 1.5, 4.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model_a2.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    model_a2.train()
    tl, c, tot = 0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model_a2(bx), by)
        loss.backward()
        nn.utils.clip_grad_norm_(model_a2.parameters(), 1.0)
        optimizer.step()
        tl += loss.item(); c += (model_a2(bx).argmax(1)==by).sum().item(); tot += by.size(0)
    scheduler.step()
    print(f"Epoch {epoch:02d} | Loss: {tl/len(train_loader):.4f} | Acc: {100*c/tot:.1f}%")


Epoch 01 | Loss: 0.2796 | Acc: 95.3%
Epoch 02 | Loss: 0.0853 | Acc: 99.7%
Epoch 03 | Loss: 0.0436 | Acc: 99.9%
Epoch 04 | Loss: 0.0430 | Acc: 99.8%
Epoch 05 | Loss: 0.0366 | Acc: 99.9%
Epoch 06 | Loss: 0.0351 | Acc: 99.9%
Epoch 07 | Loss: 0.0302 | Acc: 99.9%
Epoch 08 | Loss: 0.0274 | Acc: 99.9%
Epoch 09 | Loss: 0.0273 | Acc: 99.9%
Epoch 10 | Loss: 0.0268 | Acc: 99.9%


### Final Clinical Validation
Finally, I'm checking the results on my test patients. I'm looking for high accuracy and a strong classification report. This temporal approach is meant to be more robust for real-time monitoring where the context of previous beats matters just as much as the current one.

In [6]:
# Cell 5: Evaluation
model_a2.eval()
preds, labs = [], []
with torch.no_grad():
    for bx, by in test_loader:
        preds.extend(model_a2(bx.to(device)).argmax(1).cpu().numpy())
        labs.extend(by.numpy())
preds, labs = np.array(preds), np.array(labs)

pcls = sorted(np.unique(np.concatenate([labs, preds])))
pnames = [CLASS_NAMES[i] for i in pcls if i < len(CLASS_NAMES)]
print(f"Accuracy: {(preds==labs).mean()*100:.2f}%")
print("="*60)
print(classification_report(labs, preds, labels=pcls, target_names=pnames, zero_division=0))


Accuracy: 95.27%
              precision    recall  f1-score   support

  Normal (N)       0.94      1.00      0.97      3355
     PVC (V)       0.99      0.81      0.89      1020

    accuracy                           0.95      4375
   macro avg       0.97      0.90      0.93      4375
weighted avg       0.95      0.95      0.95      4375

